# Cadrille — Google Colab

Supports:
- **Eval**: run inference + metrics on a checkpoint (free T4 / Pro A100)
- **SFT smoke test**: 50-step training run to verify the pipeline
- **RL fine-tuning**: Dr. CPPO on H100 (Pro+ recommended)
- **Monitoring**: sync W&B offline runs

> Mount Google Drive first — data and checkpoints survive session restarts there.

In [ ]:
# ── [1] Mount Drive ──────────────────────────────────────────────────────────
# All data and checkpoints live here — they survive session restarts.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/cadrille'   # adjust if needed
DRIVE_DATA  = f'{DRIVE_ROOT}/data'
DRIVE_CKPTS = f'{DRIVE_ROOT}/checkpoints'

import os
os.makedirs(DRIVE_DATA,  exist_ok=True)
os.makedirs(DRIVE_CKPTS, exist_ok=True)
print('Drive mounted. Paths:', DRIVE_DATA, DRIVE_CKPTS)

In [ ]:
# ── [2] Clone repo ───────────────────────────────────────────────────────────
import os

REPO = '/content/cadrille'
if not os.path.exists(REPO):
    !git clone https://github.com/miachen0401/cadrille.git {REPO}
else:
    !git -C {REPO} pull   # update to latest

%cd {REPO}

In [ ]:
# ── [3] Install dependencies ─────────────────────────────────────────────────
# Pinned stack: PyTorch 2.5.1 + CUDA 12.4 + Python 3.12
# Uses uv for fast installs; version checks skip expensive steps on re-run.
# pytorch3d: built once via pip (uv can't compile CUDA), wheel cached to Drive.

import sys, subprocess, os, glob, shutil

def _run(cmd, env=None):
    """Run command, streaming output through Jupyter's stdout in real time."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, env=env)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

# ── 0. System libraries (required by OCC/CadQuery in headless environments) ──
_run(["apt-get", "install", "-y", "libgl1", "libglib2.0-0"])

# ── 1. Bootstrap uv ──────────────────────────────────────────────────────────
_uv_check = subprocess.run(["uv", "--version"], capture_output=True, text=True)
if _uv_check.returncode != 0:
    _run(["pip", "install", "uv"])
    print("uv installed ✓")
else:
    print(f"uv {_uv_check.stdout.strip()} ✓")

# Point uv at Colab's system Python (no virtualenv in Colab)
os.environ["UV_SYSTEM_PYTHON"] = "1"

def _pkg_version(pkg):
    """Return installed version string, or None."""
    r = subprocess.run(["uv", "pip", "show", pkg], capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith("Version:"):
            return line.split(":", 1)[1].strip()
    return None

def uv_install(*args):
    _run(["uv", "pip", "install"] + list(args))

def uv_uninstall(*args):
    _run(["uv", "pip", "uninstall"] + list(args))

# ── 2. PyTorch 2.5.1 + CUDA 12.4 ────────────────────────────────────────────
_torch_ver = _pkg_version("torch")
if _torch_ver == "2.5.1+cu124":
    print(f"torch {_torch_ver} ✓")
    _reinstalled_torch = False
else:
    print(f"torch {_torch_ver or '(none)'} → installing 2.5.1+cu124 ...")
    uv_uninstall("torch", "torchvision", "torchaudio", "flash-attn")
    uv_install("torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
               "--index-url", "https://download.pytorch.org/whl/cu124")
    _reinstalled_torch = True
    print("torch 2.5.1+cu124 ✓  (kernel restart needed if torch was already imported)")

# ── 3. pytorch3d — build once, cache wheel to Drive for instant future installs
_p3d = subprocess.run([sys.executable, "-c", "import pytorch3d; print(pytorch3d.__version__)"],
                      capture_output=True, text=True)
if _p3d.returncode == 0 and not _reinstalled_torch:
    print(f"pytorch3d {_p3d.stdout.strip()} ✓")
else:
    if not shutil.which("ninja"):
        uv_install("ninja")
    _ninja_ver = subprocess.run(["ninja", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"ninja {_ninja_ver} ✓")

    _drive_wheels = os.path.join(globals().get('DRIVE_ROOT', '/tmp'), 'wheels')
    os.makedirs(_drive_wheels, exist_ok=True)
    _cached = glob.glob(f'{_drive_wheels}/pytorch3d*.whl')

    if _cached and not _reinstalled_torch:
        print(f"Installing cached pytorch3d wheel from Drive (~30s) ...")
        _run([sys.executable, "-m", "pip", "install", _cached[0]])
    else:
        _njobs = str(os.cpu_count() or 8)
        print(f"Building pytorch3d from source (MAX_JOBS={_njobs}, ~8-15 min) ...")
        print("  (wheel cached to Drive — subsequent sessions skip this step)")
        _run([sys.executable, "-m", "pip", "wheel",
              "--no-deps", "--wheel-dir", _drive_wheels,
              "git+https://github.com/facebookresearch/pytorch3d.git"],
             env={**os.environ, "MAX_JOBS": _njobs, "FORCE_CUDA": "1"})
        _wheel = glob.glob(f'{_drive_wheels}/pytorch3d*.whl')[0]
        print(f"Cached: {_wheel}")
        _run([sys.executable, "-m", "pip", "install", _wheel])
    print("pytorch3d ✓")

# ── 4. flash-attn prebuilt wheel (torch 2.5 + CUDA 12 + Python 3.12) ─────────
_fa_ver = _pkg_version("flash-attn")
if _fa_ver and _fa_ver.startswith("2.7.2") and not _reinstalled_torch:
    print(f"flash-attn {_fa_ver} ✓")
else:
    print(f"flash-attn {_fa_ver or '(none)'} → installing 2.7.2.post1 ...")
    uv_install(
        "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.2.post1/"
        "flash_attn-2.7.2.post1+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl")
    print("flash-attn ✓")

# ── 5. cadquery ───────────────────────────────────────────────────────────────
# nptyping<2.5.0 uses np.bool8 which was removed in NumPy 2.0 (Python 3.12).
# Must upgrade before testing cadquery — it's a cadquery transitive dependency.
uv_install("nptyping==2.5.0")

_cq_test = subprocess.run(
    [sys.executable, "-c",
     "import cadquery as cq, json; r=cq.Workplane('XY').box(1,1,1); "
     "v,f=r.val().tessellate(0.01); print(json.dumps({'faces':len(f)}))"],
    capture_output=True, text=True)
if _cq_test.returncode == 0:
    print(f"cadquery ✓  {_cq_test.stdout.strip()}")
else:
    print(f"cadquery test failed after nptyping fix — reinstalling cadquery stack...")
    uv_install("cadquery-ocp==7.7.2", "cadquery==2.4.0")
    _cq_test2 = subprocess.run(
        [sys.executable, "-c",
         "import cadquery as cq, json; r=cq.Workplane('XY').box(1,1,1); "
         "v,f=r.val().tessellate(0.01); print(json.dumps({'faces':len(f)}))"],
        capture_output=True, text=True)
    if _cq_test2.returncode != 0:
        raise RuntimeError(f"cadquery still broken:\n{_cq_test2.stderr}")
    print(f"cadquery ✓  {_cq_test2.stdout.strip()}")

# ── 6. Remaining deps ─────────────────────────────────────────────────────────
uv_install(
    "transformers==4.50.3", "accelerate==0.34.2", "qwen-vl-utils==0.0.10",
    "trimesh==4.5.3", "open3d", "scipy==1.14.1",
    "wandb", "tqdm", "pyyaml")

# ── 7. Verify ─────────────────────────────────────────────────────────────────
import torch, flash_attn, pytorch3d
p = torch.cuda.get_device_properties(0)
print(f"\nGPU        : {p.name}  {p.total_memory // 1024**3} GB")
print(f"CUDA       : {torch.version.cuda}  |  torch {torch.__version__}  |  Python {sys.version.split()[0]}")
print(f"flash-attn : {flash_attn.__version__}")
print(f"pytorch3d  : {pytorch3d.__version__}")

In [ ]:
# ── [4] Link data & checkpoints from Drive ───────────────────────────────────
import os

if not os.path.exists('data'):
    os.symlink(DRIVE_DATA, 'data')

if not os.path.exists('checkpoints'):
    os.symlink(DRIVE_CKPTS, 'checkpoints')

print('data/      →', os.listdir('data') or '(empty)')
print('checkpoints/ →', os.listdir('checkpoints') or '(empty)')

In [ ]:
# ── [5] Download training & validation data (run once, persists on Drive) ──────
# Downloads DeepCAD test meshes (RL training + validation) and Fusion360 test
# meshes (validation only). Each is a separate HuggingFace dataset repo.
# Skip this cell if both folders already exist in data/.

import os

if not os.path.exists('data/deepcad_test_mesh'):
    # ~1 GB — DeepCAD test split (STL meshes); used as RL training prompts + eval
    !huggingface-cli download maksimko123/deepcad_test_mesh \
        --repo-type dataset \
        --local-dir data/deepcad_test_mesh

if not os.path.exists('data/fusion360_test_mesh'):
    # ~200 MB — Fusion360 test split; used for cross-dataset validation only
    !huggingface-cli download maksimko123/fusion360_test_mesh \
        --repo-type dataset \
        --local-dir data/fusion360_test_mesh

print('Data ready:')
!ls data/deepcad_test_mesh | head -5
!echo "  ... $(ls data/deepcad_test_mesh | wc -l) deepcad meshes"
!echo "  ... $(ls data/fusion360_test_mesh | wc -l) fusion360 meshes"

In [ ]:
# ── [6] Download SFT checkpoint (run once, persists on Drive) ─────────────────
# Starting point for RL fine-tuning.
# We check for the actual weight file, not just the directory — snapshot_download
# can create an empty directory if it fails partway through.

import os, glob

SFT_CKPT = 'checkpoints/cadrille-sft'

# A valid checkpoint must contain at least one weight shard
_weights = (glob.glob(f'{SFT_CKPT}/model*.safetensors') +
            glob.glob(f'{SFT_CKPT}/pytorch_model*.bin'))

if _weights:
    print(f'Checkpoint already downloaded: {SFT_CKPT}')
    print(f'  weight files: {[os.path.basename(w) for w in _weights]}')
else:
    print(f'Downloading cadrille SFT checkpoint → {SFT_CKPT}  (~4–5 GB, a few min) ...')
    # Remove stale/empty directory first so huggingface-cli doesn't skip files
    if os.path.exists(SFT_CKPT):
        import shutil
        shutil.rmtree(SFT_CKPT)
    !huggingface-cli download maksimko123/cadrille \
        --repo-type model \
        --local-dir {SFT_CKPT}
    print(f'Done. Contents:')
    !ls -lh {SFT_CKPT}

In [ ]:
# ── [7] W&B login ────────────────────────────────────────────────────────────
import wandb
wandb.login()   # paste your API key from wandb.ai/authorize

In [ ]:
# ── [8] RL fine-tuning ────────────────────────────────────────────────────────
# Auto-selects config based on GPU:
#   H100  80 GB → configs/rl/h100.yaml   G=16, batch_updates=3  (~8-12 steps/min)
#   A100  80 GB → configs/rl/h100.yaml   (same memory budget)
#   A100  40 GB → configs/rl/a100.yaml   G=8,  batch_updates=2  (~5-8  steps/min)
#   RTX 4080 16 GB → configs/rl/4080.yaml G=4, sequential       (~2-4  steps/min)
#
# Checkpoints saved to Drive every 500 steps; eval every 500 steps.
# W&B run ID saved to checkpoints/<run_name>/wandb_run_id.txt for resume.

import torch

_gpu_name = torch.cuda.get_device_name(0)
_vram_gb  = torch.cuda.get_device_properties(0).total_memory // 1024**3
print(f"Detected: {_gpu_name}  {_vram_gb} GB")

if _vram_gb >= 70:          # H100 80 GB or A100 80 GB
    CONFIG = 'configs/rl/h100.yaml'
elif _vram_gb >= 35:        # A100 40 GB
    CONFIG = 'configs/rl/a100.yaml'
else:                       # RTX 4080 / other 16 GB
    CONFIG = 'configs/rl/4080.yaml'

print(f"Using config: {CONFIG}")

RUN_NAME = 'cadrille-rl-v1'   # change for a new run; keep same to resume

!python rl/train.py \
    --config  {CONFIG} \
    --run-name {RUN_NAME}

In [ ]:
# ── [9] Resume after session timeout ─────────────────────────────────────────
# Colab sessions timeout (~12h Pro, ~24h Pro+).
# When your session restarts:
#   1. Re-run cells [1] – [7] (Drive mount, clone, install, link, login)
#   2. Run this cell to continue from the latest saved checkpoint.
#
# The step counter resumes at the checkpoint step (no duplicate steps logged).
# W&B attaches to the original run via the saved run ID in wandb_run_id.txt.

import glob, torch

RUN_NAME = 'cadrille-rl-v1'   # must match the original run

# Auto-select same config as the original run
_vram_gb = torch.cuda.get_device_properties(0).total_memory // 1024**3
if _vram_gb >= 70:
    CONFIG = 'configs/rl/h100.yaml'
elif _vram_gb >= 35:
    CONFIG = 'configs/rl/a100.yaml'
else:
    CONFIG = 'configs/rl/4080.yaml'

ckpt_dirs = sorted(glob.glob(f'checkpoints/{RUN_NAME}/checkpoint-[0-9]*'),
                   key=lambda p: int(p.rsplit('-', 1)[-1]))
if not ckpt_dirs:
    print('No checkpoints found — start a fresh run with cell [8] instead.')
else:
    latest = ckpt_dirs[-1]
    step   = int(latest.rsplit('-', 1)[-1])
    print(f'GPU: {torch.cuda.get_device_name(0)}  {_vram_gb} GB  →  {CONFIG}')
    print(f'Resuming from: {latest}  (step {step} → 50000)')
    !python rl/train.py \
        --config           {CONFIG} \
        --run-name         {RUN_NAME} \
        --checkpoint-path  {latest}